# Track B — Qwen2.5-VL-3B QLoRA (Colab, 주 실행처) — **_0705 최종본**

**이 노트북은 2026-07-05 LB 0.71 제출을 만든 확정 파이프라인이다.** 산출물 이름은 전부 `_0705` 접미사로 Drive에 동결 (`qwen25vl3b_merged_0705`, `raw_*_0705.jsonl`, `submission_0705.csv`).

런타임 → 런타임 유형 변경 → **A100 GPU** (T4도 가능: B2의 OVERRIDES만 조정, 추론은 fp16 폴백).

**Colab 주의**: 세션이 예고 없이 끊기고 `/content`는 휘발된다 → 체크포인트·raw·모델은 전부 Drive에 저장. 재실행 시 `resume=True`.

사전 준비 (Drive의 `MyDrive/snuai/`에 업로드):
1. `snuai_code_0705.zip` — 로컬에서 `python -m cloud.pack_code` 산출물 (2026-07-05 스냅샷: bf16 자동 dtype, 손상 이미지 스킵, 새 집계 정책 포함)
2. `access_token` — kaggle.com/settings → API → **Generate New Token**으로 화면에 표시된 토큰 문자열을 확장자 없는 텍스트 파일로 저장한 것 (2026 API v2: 구식 kaggle.json 아님)

실행 순서: A1→A2→A3→A4(재시작)→A1·A3→B1(스모크)→B2(본학습)→B3(병합)→**B4(병합 검증, 필수)**→C1(val)→C2(test·제출)

In [ ]:
# A1) Drive 마운트 + 코드 압축 해제 (기존 코드 청소 후 — 구버전 파일 잔류 방지)
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/code && unzip -qo /content/drive/MyDrive/snuai/snuai_code_0705.zip -d /content/code
import sys; sys.path.insert(0, '/content/code')

In [ ]:
# A2) 대회 데이터 — Kaggle API v2 인증 (단일 토큰 방식)
# 토큰 발급: kaggle.com/settings → API → "Generate New Token" → 화면에 뜬 문자열 복사
# (구식 kaggle.json은 2026 API 개편으로 401 거부됨)
SLUG = 'snuaichallenge'
!pip install -qU kaggle  # v2 CLI 보장 (access_token 지원)

# 방법 1(권장): 토큰 문자열을 확장자 없는 파일 'access_token'으로 저장해 Drive의 snuai/에 업로드
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/snuai/access_token ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
# 방법 1이 번거로우면: 아래 두 줄 주석 해제 후 토큰을 직접 붙여넣기 (노트북 공유 금지!)
# TOKEN = '여기에-토큰-붙여넣기'
# !mkdir -p ~/.kaggle && printf %s {TOKEN} > ~/.kaggle/access_token

!kaggle competitions download {SLUG} -p /content
!unzip -qo /content/{SLUG}.zip -d /content/data
# API가 계속 거부되면 최후 수단: Drive에 올린 snuaichallenge.zip 사용
# !unzip -qo /content/drive/MyDrive/snuai/snuaichallenge.zip -d /content/data
!ls /content/data

In [ ]:
# A3) 경로 설정 — train.csv 위치 자동 탐지 (중첩 폴더 대응)
import os, glob, shutil
hits = sorted(glob.glob('/content/data/train.csv')
              + glob.glob('/content/data/*/train.csv')
              + glob.glob('/content/data/*/*/train.csv'))
assert hits, 'train.csv를 찾지 못함 — A2의 압축 해제 결과 확인'
DATA_ROOT = os.path.dirname(hits[0])
print('DATA_ROOT:', DATA_ROOT)

yaml_text = '\n'.join([
    f'data_dir: {DATA_ROOT}',
    f'train_csv: {DATA_ROOT}/train.csv',
    f'test_csv: {DATA_ROOT}/test.csv',
    f'sample_submission: {DATA_ROOT}/sample_submission.csv',
    f'train_image_dir: {DATA_ROOT}/train',
    f'test_image_dir: {DATA_ROOT}/test',
    'models_dir: /content/models',
    'outputs_dir: /content/outputs',
    'reports_dir: /content/reports',
])
open('/content/paths.yaml', 'w').write(yaml_text)
os.environ['SNUAI_PATHS_CONFIG'] = '/content/paths.yaml'
os.makedirs('/content/outputs', exist_ok=True)
# split.csv는 코드 번들에 있으므로 outputs_dir로 복사 (--fold val 경로가 참조)
shutil.copy('/content/code/outputs/split.csv', '/content/outputs/split.csv')

from src.data.loader import load_split
print('train rows:', len(load_split('train')), '| test rows:', len(load_split('test')))

In [ ]:
# A4) 설치 + 재현성 증빙
# ⚠️ 이 셀을 처음 실행한 뒤에는 반드시 [런타임 → 세션 다시 시작] 후 A1, A3만 재실행하고 B1로.
#    (unsloth가 pyarrow 등을 교체하므로 재시작 없이는 바이너리 불일치 에러 발생)
!pip install -q unsloth imagehash
import subprocess, sys
open('/content/drive/MyDrive/snuai/pip_freeze_colab.txt','w').write(
    subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True).stdout)
print('설치 완료 — 런타임을 재시작한 뒤 A1, A3 재실행 후 B1을 진행하세요.')

In [ ]:
# B1) 스모크 (32샘플) — 장기 학습 전 필수
import sys
sys.path.insert(0, '/content/code')  # 재실행 대비
from cloud.train_unsloth import run
SFT = '/content/code/outputs/sft_train.jsonl'
OUT = '/content/drive/MyDrive/snuai/outputs/qwen25vl3b_0705'  # Drive = 세션 휘발 대비
run('/content/code/configs/sft_qwen.yaml', SFT, DATA_ROOT, limit=32, output_dir=OUT + '_smoke')

In [ ]:
# B2) 본 학습 — 첫 실행 resume=False, 세션 끊긴 뒤 재실행은 resume=True
# GPU별 배치 (유효 배치 16 고정): T4 -> {'per_device_batch': 1, 'grad_accum': 16}
#                                A100/G4 -> {'per_device_batch': 4, 'grad_accum': 4}
#                                L4 -> {'per_device_batch': 2, 'grad_accum': 8}
OVERRIDES = {'per_device_batch': 4, 'grad_accum': 4}  # A100 기준
lora_dir = run('/content/code/configs/sft_qwen.yaml', SFT, DATA_ROOT, resume=False,
               output_dir=OUT, train_overrides=OVERRIDES)
print('LoRA saved:', lora_dir)

In [ ]:
# B3) 병합 저장 (Drive) — 추론은 플레인 transformers 경로 사용
# ⚠️ save_pretrained_merged는 vision 모델에서 LoRA를 병합하지 않고 베이스만 저장
#    (unsloth#1352, 2026-07-05 실측: val EM이 랜덤으로 추락) → peft 표준 병합 사용.
from unsloth import FastVisionModel
OUT = '/content/drive/MyDrive/snuai/outputs/qwen25vl3b_0705'
MERGED = '/content/drive/MyDrive/snuai/qwen25vl3b_merged_0705'
model, processor = FastVisionModel.from_pretrained(OUT + '/lora', load_in_4bit=False)
merged = model.merge_and_unload()
merged.save_pretrained(MERGED)
processor.save_pretrained(MERGED)
print('merged saved:', MERGED, '— B4 병합 검증을 통과한 뒤에만 추론에 사용')

In [ ]:
# C1) val 리허설 (제출 전 필수) — raw는 Drive에 직접 저장 (세션 휘발 대비)
# 0705 실측: EM 0.4439 (분산 게이트 on / identity-ban off 정책, aggregate.py 기본값)
MODEL = '/content/drive/MyDrive/snuai/qwen25vl3b_merged_0705'
!cd /content/code && python -m src.infer.predict --model $MODEL --split train --fold val \
    --style mid --tta 4 --batch 16 --out /content/drive/MyDrive/snuai/raw_val_0705.jsonl
!cd /content/code && python -m src.infer.aggregate --raw /content/drive/MyDrive/snuai/raw_val_0705.jsonl \
    --out /content/outputs/pred_val.csv
!cd /content/code && python -m src.eval.em --pred /content/outputs/pred_val.csv --fold val

In [ ]:
# C2) test 추론 → 검증된 submission — raw도 Drive에 직접 (재집계 시 재추론 방지)
# 손상 이미지로 중단되면 데이터 재압축해제 후 같은 명령 재실행 (완료분 자동 스킵).
# 0705 실측: LB 0.71 (2026-07-05 제출)
!cd /content/code && python -m src.infer.predict --model $MODEL --split test \
    --style mid --tta 4 --batch 16 --out /content/drive/MyDrive/snuai/raw_test_0705.jsonl
!cd /content/code && python -m src.infer.aggregate --raw /content/drive/MyDrive/snuai/raw_test_0705.jsonl \
    --submission /content/drive/MyDrive/snuai/submission_0705.csv
print('Drive의 snuai/submission_0705.csv를 다운로드해 Kaggle에 제출')

In [ ]:
# C2) test 추론 → 검증된 submission.csv (Drive에 복사 후 Kaggle 웹에서 제출)
!cd /content/code && python -m src.infer.predict --model $MODEL --split test \
    --style mid --tta 4 --batch 16 --out /content/outputs/raw_test.jsonl
!cd /content/code && python -m src.infer.aggregate --raw /content/outputs/raw_test.jsonl \
    --submission /content/outputs/submission.csv
!cp /content/outputs/submission.csv /content/drive/MyDrive/snuai/submission.csv
print('Drive의 snuai/submission.csv를 다운로드해 Kaggle에 제출')